In [1]:
from phase_II.utils.helpers import usual_plot_updated
from useful.helpers import *
%matplotlib tk
seed = 42
jax.config.update("jax_enable_x64", True)
key = jax.random.PRNGKey(seed)

wigner_function_from_inference, t, f = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")
t = t + 16

wigner_function_from_inference = np.fft.fftshift(wigner_function_from_inference)
f = np.fft.fftshift(f)

In [2]:
smooth_wigner = smooth_matrix(wigner_function_from_inference, smoothing_lvl=5, mode="gaussian")

In [3]:
visualize_stress(wigner_function_from_inference, rows=f, cols=t, smooth=True, xlim=(16.2, 16.5), ylim=(-700,700), save_fig=True)

/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:65: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [4]:
white_noise = np.random.standard_normal(len(f))
white_noise_stress, _, _ = Stress_re(white_noise, time=t, supress_print=True)
smooth_white_stress = smooth_matrix(white_noise_stress, smoothing_lvl=5, mode="gaussian")

In [5]:
visualize_stress(1600*jnp.abs(white_noise_stress), rows=f, cols=t, smooth=False)

/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [6]:
plt.plot(t, np.sum(jnp.abs(smooth_wigner), axis=0)-np.sum(jnp.abs(white_noise_stress), axis=0))
usual_plot()

In [7]:
# pos_definite_wigner = jnp.abs(smooth_wigner)
pos_definite_wigner = jnp.abs(smooth_wigner)

In [8]:
# pos_definite_wigner: shape (n_rows, n_cols)
n_rows, n_cols = pos_definite_wigner.shape

# Example time axis
# t must have length n_cols
# t = np.linspace(0, 10, n_cols)  # or your real time axis

# Define the column bins
n_bins = 200
col_edges = np.linspace(0, n_cols, n_bins + 1, dtype=int)

binned_sum = np.zeros(n_bins)
t_binned = np.zeros(n_bins)

for i in range(n_bins):
    start = col_edges[i]
    end = col_edges[i+1]
    binned_sum[i] = pos_definite_wigner[:, start:end].sum()
    t_binned[i] = t[start:end].mean()  # mean time in this bin

print("t discretization: ", t_binned[1]-t_binned[0])

plt.plot(t_binned, binned_sum)
plt.xlabel('Time')
plt.ylabel('Sum over rows')
plt.show()


t discretization:  0.0098876953125


In [9]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter

# Example square image
A = pos_definite_wigner

patch_size = 5  # size of sliding patch
# compute local mean
local_mean = uniform_filter(A, size=patch_size, mode='constant')

# coordinates of the patch centers
n_rows, n_cols = A.shape
offset = patch_size // 2
x_centers = np.arange(offset, n_cols - offset)
y_centers = np.arange(offset, n_rows - offset)

# extract the central region corresponding to patch centers
center_means = local_mean[offset:n_rows-offset, offset:n_cols-offset]

# plot
X, Y = np.meshgrid(x_centers, y_centers)
plt.scatter(X, Y, c=center_means, cmap='viridis')
plt.colorbar(label='Patch mean')
plt.xlabel('Column index')
plt.ylabel('Row index')
plt.show()


In [10]:
import numpy as np
from scipy.ndimage import label

A = pos_definite_wigner

# Step 1: threshold to define structures
threshold = 2000  # adjust to select "strong" structures
binary = A > threshold

# Step 2: label connected components
labeled, n_features = label(binary)

# Step 3: remove small components
min_size = 10  # minimum number of pixels
component_sizes = np.bincount(labeled.ravel())
size_mask = component_sizes >= min_size
size_mask[0] = 0  # background

# Step 4: filtered array keeping original A values
filtered = A * size_mask[labeled]


In [11]:
# visualize_stress(filtered, rows=f, cols=t, smooth=False)

# pos_definite_wigner_copy = np.array(pos_definite_wigner.copy())
pos_definite_wigner_copy = np.array(smooth_wigner.copy().real**2)
indcs = np.where(f == 0)
slice_along_f_0 = pos_definite_wigner_copy[indcs, :][0][0]

visualize_stress(pos_definite_wigner_copy, rows=f, cols=t, smooth=False, xlim=(16.2, 16.5), ylim=(-700,700), save_fig=True)

/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:65: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [12]:
# plt.plot(t, np.trapz(filtered, axis=0, dx=f[1]-f[0]))
plt.plot(t, np.sum(jnp.abs(smooth_wigner), axis=1))
plt.show()

In [13]:
f[1]-f[0]

np.float64(0.5000610426077401)

In [14]:
plt.plot(t, np.sum(pos_definite_wigner_copy, axis=0))
usual_plot(xl="Time [sec]", yl="Stress squared", title=r"$\sum_f S(t,f)^2$",save_fig=True)

In [15]:
# gradient_stress = np.gradient(slice_along_f_0, 1)
plt.plot(t, slice_along_f_0)
usual_plot(xl="Time [sec]", yl="Stress squared", title=r"$S(t,f=0)^2$",save_fig=True)

time_left = 0.292
time_right = 0.462

In [46]:
time_indcs = (t > time_left) & (t < time_right)
freq_indcs = (f < 0)
reduced_freqs = f[freq_indcs]

# all_stress_values_under_consideration = pos_definite_wigner_copy[:, time_indcs]
all_stress_values_under_consideration = pos_definite_wigner_copy[np.ix_(freq_indcs, time_indcs)]

In [ ]:
all_stress_values_under_consideration = np.flip(all_stress_values_under_consideration, axis=0)

def show_non_square_mat(mat):

    plt.matshow(mat, fignum=1, aspect="auto")  # aspect='auto' lets it stretch to fit
    plt.colorbar()
    plt.show()

show_non_square_mat(all_stress_values_under_consideration)

In [ ]:
all_stress_values_under_consideration

In [ ]:
all_stress_values_under_consideration_summed_along_columns = np.max(all_stress_values_under_consideration, axis=1)

In [ ]:
_ = plt.figure()
plt.plot(reduced_freqs[::-1], all_stress_values_under_consideration_summed_along_columns)

### Therefore,

a GW is at approximately between f = (-290, 290) and times (0.26, 0.5)

In [3]:
wigner_function_from_inference, t, f = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")

wigner_function_from_inference = np.fft.fftshift(wigner_function_from_inference)
f = np.fft.fftshift(f)

smooth_wigner = smooth_matrix(wigner_function_from_inference, smoothing_lvl=5, mode="gaussian")
mat = smooth_wigner.real.copy()

f_cut = (f > -290) & (f < 290)
t_cut = (t > 0.26) & (t < 0.5)

reduced_frequencies = f[f_cut]
reduced_times = t[t_cut]

reduced_frequencies_dft_order = np.fft.ifftshift(reduced_frequencies)

print(reduced_frequencies_dft_order)

mat_cut = np.fft.ifftshift(mat[np.ix_(f_cut, t_cut)])

visualize_stress(np.fft.ifftshift(mat_cut), rows=np.fft.fftshift(reduced_frequencies_dft_order), cols=reduced_times, smooth=False)

[ 0.          0.50006104  1.00012209 ... -1.50018313 -1.00012209
 -0.50006104]


In [47]:
def show_non_square_mat(mat):

    plt.matshow(mat, fignum=1, aspect="auto")  # aspect='auto' lets it stretch to fit
    plt.colorbar()
    plt.show()

show_non_square_mat(np.fft.ifftshift(mat_cut))
# show_non_square_mat(mat_cut)

In [48]:
nrt_strain_values = np.loadtxt("../../data/data_txt/num_rel_template_strain_values.txt") * 1e19
nrt_time_values = np.loadtxt("../../data/data_txt/num_rel_template_time_values.txt")
nrt_time_values = nrt_time_values - nrt_time_values[0]

nrt_stress, time_nrt, freq_nrt  = Stress_re(xi=nrt_strain_values, time=nrt_time_values)

visualize_stress(nrt_stress, rows=freq_nrt, cols=time_nrt, smooth=False)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.6039972777988179e-24) 
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [6]:
visualize_stress(nrt_stress, rows=freq_nrt, cols=time_nrt, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [50]:
def _find_nearest_subdiagonal_matches(a, b, rtol=.1, print_arrays=False):
    """

    a = np.linspace(-2, 7, 10)
    b = np.linspace(-3, 3, 8)

    _,_,_ = _find_nearest_subdiagonal_matches(a, b, print_arrays=True, rtol=.1)

    Returns indices and arrays a_prime, b_prime such that

        a_prime = 1/2 * b_prime

    in the given relative tolerance.

    Also returns the index array, the array of found minima and mean minimum. The smaller the mean
    minimum is, the better the array values match.

    If the arrays don't match exactly, or towards the end of the arrays, the residual between a and 1/2b will not have
    a precise zero match. In that case, accept the index if a = rtol * 1/2b at that point.

    Square Example:
        b  [0. 1. 2. 3.]
        a  [0. 1. 2. 3.]

        >> _find_nearest_subdiagonal_matches(a,b, rtol=.1)
        >>  a_prime:        [0. 1.]
            1/2 * b_prime:  [0. 1.]

        >> _find_nearest_subdiagonal_matches(a,b, rtol=.5)
        >>  a_prime:        [0. 1. 2. 3.]
        >>  1/2 * b_prime:  [0.  1.  1.5 1.5]  ( i.e.: 1.5/2 <= 50% and 1.5/3 <= 50%)

    Non-square example:

        a = [ 0.          2.22222222  4.44444444  6.66666667  8.88888889 11.11111111 13.33333333 15.55555556 17.77777778 20.        ]
        b = a[-1] # ==> 20 = 1/2 * 10 is missing, so should not be in the primed arrays

        >> a_prime [0.         2.22222222 4.44444444 6.66666667 8.88888889]
        >> b_prime [0.         2.22222222 4.44444444 6.66666667 8.88888889]

        whereas if b == a:

        >> a_prime: 		 [ 0.          2.22222222  4.44444444  6.66666667  8.88888889 11.11111111]
        >> 1/2 * b_prime: 	 [ 0.          2.22222222  4.44444444  6.66666667  8.88888889 10.        ].

    :param a:       np.ndarray
    :param b:       np.ndarray
    :param rtol:    float

    :return:    INDCS, VECS, MIN
                >> a_indcs = INDCS[0]
                >> b_indcs = INDCS[1]

                >> a_prime = VECS[0] = a[a_indcs]
                >> b_prime = VECS[1] = b[b_indcs]

                >> rel_min_vec = MIN
                >> abs_min_vec = a * MIN
    """

    dense_a_matrix = np.array([a]*len(b)) # [[a], [a], [a], ...]
    dense_half_b_matrix = 1/2 * np.array([b]*len(a)).T # 1/2 * [[b1 b1 b2 ... ]. [b2 b2 b2 ...]] = [ [b1 b2 b3 ...] column f length times]
    residual =  np.abs(dense_a_matrix - dense_half_b_matrix) #  [[a - 1/2 b1], [a - 1/2 b2], [a - 1/2 b3], ...]

    indcs = np.argmin(residual, axis=0)  # indcs such that 1/2 * b[indcs] = a.
    minima_vector = np.min(residual, axis=0)
    relative_minima_vector = minima_vector / (np.abs(a)+1e-10)

    half_b_array = 1/2*b[indcs]

    threshhold = np.where(relative_minima_vector < rtol)
    indcs = indcs[threshhold]

    b_indcs = indcs
    a_indcs = threshhold

    b_prime = b[b_indcs]
    a_prime = a[a_indcs]  # because if you print minima vector there will be some 0's and a_prime needs to be evaluated at those 0's. And if the 0's don't start
    # at index 0, which is the case for negative numbers, a[threshhold] is not the same as a[:len(b_prime)]...

    if print_arrays:

        print(f"Original a (len {len(a)}) array:\n",a)
        print(f"\nOriginal b (len ({len(b)})) array:\n",b)

        print(f"\na_prime (len {len(a_prime)}): \t\t",  a_prime)
        print(f"1/2 * b_prime (len {len(b_prime)}): \t", 1/2 *b_prime,"\n")

    INDCS = (a_indcs[0], b_indcs)  # remove extra tuple wrapper
    VECS = (a_prime, b_prime)
    MIN = relative_minima_vector[threshhold]
    return INDCS, VECS, MIN


def cgpt_is_standard_dft_order(f, atol=1e-12):
    """
    Check if a frequency array is in standard DFT order:
    first half: 0 -> positive frequencies
    second half: negative frequencies increasing toward zero

    Parameters
    ----------
    f : np.ndarray
        Frequency array
    atol : float
        Absolute tolerance for numerical comparisons

    Returns
    -------
    bool
        True if f is in standard DFT order
    """
    N = len(f)
    # split positive and negative halves
    pos_half = f[:N//2 + N%2]
    neg_half = f[N//2 + N%2:]

    # positive half: starts at 0, strictly increasing
    check_pos = np.all(pos_half >= -atol) and np.all(np.diff(pos_half) > -atol)
    # negative half: strictly negative, increasing
    check_neg = np.all(neg_half < atol) and np.all(np.diff(neg_half) > -atol)

    return check_pos and check_neg


def invert_wigner_function_re(S_mat, frequency_array, time_array, xi_tilde_0=None, debug_plot=False):
    """
    :ToDo: Use jnp instead of np
    Notes:

        - S_mat can be square or not
        - time_array and the corresponding S_mat column axis must be in standard dft order, i.e. monotonic in time, which it should be.
        - frequency_array and the corresponding S_mat row axis must be in standard dft order. It needs to be in standard dft order
          because the computed q array is as well.

    :param S_mat:
    :param frequency_array:
    :param time_array:
    :param xi_tilde_0:
    :param debug_plot:      Plot the subdiagonal found for which f ~ 1/2 q.
    :return:
    """

    h_fw = lambda inp: jnp.fft.fft(inp, axis=1, norm='ortho')  # domain=(h_dom, r_dom) --> (h_dom, h_dom)
    h_bw = lambda inp: jnp.fft.ifft(inp, norm='ortho')  # domain=(h_dom) --> (r_dom)

    f = np.array(frequency_array)
    t = np.array(time_array)
    dt = t[1] - t[0]
    q = np.fft.fftfreq(n=len(t), d=dt)
    S_ft = np.array(S_mat)


    if S_ft.shape[0] == S_ft.shape[1]:
        assert np.allclose(f, q, atol=0, rtol=.01)
        assert f[0] == 0        # DFT ordering check

    assert cgpt_is_standard_dft_order(f)
    assert cgpt_is_standard_dft_order(q)

    S_fq = h_fw(S_ft)  # rows: f, colums: q

    rel_tol = .01
    (f_INDCS, q_INDCS), (f_NEW, q_NEW), _ = _find_nearest_subdiagonal_matches(a=f, b=q, print_arrays=False, rtol=rel_tol)

    if debug_plot:
        plt.plot(f_NEW, label="f new")
        plt.plot(1/2*q_NEW, label="1/2 * q new")
        plt.legend()
        plt.show()

    assert np.allclose(f_NEW, q_NEW*1/2, atol=0, rtol=rel_tol*2)

    Sigma_f = S_fq[f_INDCS, q_INDCS]  # elementwise indices!
    # Sigma_f = S_fq[np.ix_(f_INDCS, q_INDCS)]  # outer product of indices

    xi = h_bw(Sigma_f)
    df = q_NEW[1]-q_NEW[0]
    dual_times = np.arange(len(xi)) * 1 / (len(xi) * df)

    if xi_tilde_0 is not None:
        xi = xi / xi_tilde_0.conj()

    return dual_times, xi.real

In [51]:
reconstructed_times, reconstructed_nrt = invert_wigner_function_re(S_mat=nrt_stress, frequency_array=freq_nrt, time_array=time_nrt)

In [52]:
scaled_reconstructed_nrt = reconstructed_nrt * max(nrt_strain_values)/max(reconstructed_nrt)

plt.plot(nrt_time_values, nrt_strain_values, label=r"Original $\xi$")
plt.plot(reconstructed_times+1, scaled_reconstructed_nrt, label="My inversion of the wigner function")
plt.legend()
usual_plot()

In [9]:
k_nrt, ps_nrt = power_analyze_re(nrt_time_values, nrt_strain_values)
k_inverted, ps_inverted = power_analyze_re(reconstructed_times, scaled_reconstructed_nrt)

plt.plot(k_nrt, ps_nrt, label="PSD from nrt")
plt.plot(k_inverted, ps_inverted, label="PSD from inverse wigner(wigner(nrt))")
plt.loglog()
usual_plot(xl="Frequencies ", yl="Power")

### Now let's try it out with the cut version

In [10]:
reconstructed_times_cut_mat, reconstructed_xi_cut_mat = invert_wigner_function_re(S_mat=jnp.abs(mat_cut), frequency_array=reduced_frequencies_dft_order, time_array=reduced_times,
                                                                                  debug_plot=True)

In [12]:
plt.plot(reconstructed_times_cut_mat, reconstructed_xi_cut_mat)

In [79]:
tmp_a, tmp_b = power_analyze_re(reconstructed_times_cut_mat, reconstructed_xi_cut_mat)

In [82]:
# plt.loglog(tmp_a, tmp_b)
plt.plot(k_inverted, ps_inverted, label="PSD from inverse wigner(wigner(nrt))")
plt.loglog()

[]

In [78]:
def sample_from_ps(ps):
    r"""
    :param ps:                  The power spectrum used for the generation of data points.
    :return:
    """
    N = len(ps)
    xi = np.random.standard_normal(N)
    # xi = np.random.normal(loc=10, size=N, scale=5)
    amp = jnp.sqrt(ps)
    res = amp * xi
    return np.fft.ifft(res, norm="ortho").real

signal_sample = sample_from_ps(ps_nrt)

In [472]:
g = np.exp(-0.5 * ((nrt_time_values-1.25)/0.02)**2)
plt.plot(nrt_time_values, signal_sample*g)

In [358]:
visualize_stress(wigner_function_from_inference, f, t, smooth=True)

In [376]:
visualize_stress(nrt_stress, freq_nrt, time_nrt, smooth=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [11]:
nrt_mat = nrt_stress.real.copy()

f_nrt_cut = (freq_nrt < -290) | (freq_nrt > 290)
t_nrt_cut = (time_nrt < 1.26) | (time_nrt > 1.5)

# reduced_nrt_frequencies = freq_nrt[f_nrt_cut]
# reduced_nrt_times = time_nrt[t_nrt_cut]

nrt_mat_cut = np.array(nrt_mat.copy())
# nrt_mat_cut[np.ix_(f_nrt_cut, t_nrt_cut)] = 0
nrt_mat_cut[f_nrt_cut, :] = 0
nrt_mat_cut[:, t_nrt_cut] = 0

visualize_stress(nrt_mat_cut, rows=freq_nrt, cols=time_nrt, smooth=False)

# Original cut...
# nrt_mat = nrt_stress.real.copy()
#
# f_nrt_cut = (freq_nrt > -290) & (freq_nrt < 290)
# t_nrt_cut = (time_nrt > 1.26) & (time_nrt < 1.5)
#
# reduced_nrt_frequencies = freq_nrt[f_nrt_cut]
# reduced_nrt_times = time_nrt[t_nrt_cut]
#
# nrt_mat_cut = nrt_mat[np.ix_(f_nrt_cut, t_nrt_cut)]
#
# visualize_stress(nrt_mat_cut, rows=reduced_nrt_frequencies, cols=reduced_nrt_times, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [12]:
tmp_t, tmp_xi = invert_wigner_function_re(S_mat=nrt_mat_cut, frequency_array=freq_nrt, time_array=time_nrt, debug_plot=False)

In [13]:
plt.plot(tmp_t, tmp_xi)
plt.plot(reconstructed_times, reconstructed_nrt)

### Now lets mask everything outside the zero of region in the noisy wigner function to 0

In [56]:
wigner_function_from_inference, t_mof, f_mof = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")
t_mof = t_mof + 15
mat_of_interest = np.array(wigner_function_from_inference.copy())  # abbrev: "mof" for matrix of interest
mat_of_interest = smooth_matrix(mat_of_interest, 5, "gaussian")

visualize_stress(mat_of_interest, rows=f_mof, cols=t_mof, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [61]:
wigner_function_from_inference, t_mof, f_mof = unpickle_me_this("/Users/iason/PycharmProjects/STRAIN/phase_II/wigner_result_pipe_2.pickle")
mat_of_interest = np.array(wigner_function_from_inference.copy())  # abbrev: "mof" for matrix of interest
mat_of_interest = smooth_matrix(mat_of_interest, 5, "gaussian")


f_mof_cut = (f_mof < -290) | (f_mof > 290)
t_mof_cut = (t_mof < 1.3) | (t_mof > 1.45)
_f_mof_cut_signal = (f_mof > -290) & (f_mof < 290)

reduced_mof_frequencies = f_mof[f_mof_cut]
reduced_mof_times = t_mof[t_mof_cut]

mof_cut = np.array(mat_of_interest.copy())
# mof_cut[f_mof_cut, :] = 0.
mof_cut[:, t_mof_cut] = 0.

mof_cut_time_column_not_signal = np.array(mat_of_interest.copy())
mof_cut_time_column_not_signal[:, t_mof_cut] = 0.
mof_cut_time_column_not_signal[_f_mof_cut_signal, :] = 0.

myarr = mof_cut_time_column_not_signal.flatten()
anti_stress_energy_cut = np.mean(myarr[myarr > 1000])

mof_cut_time_column_not_signal[mof_cut_time_column_not_signal < anti_stress_energy_cut] = 0

# stress_energy_cut = anti_stress_energy_cut
stress_energy_cut = 1000
mof_cut[jnp.abs(mof_cut) < stress_energy_cut] = 0

print("Applied stress energy cut: ", stress_energy_cut)

mof_cut = smooth_matrix(mof_cut, 5, "gaussian")
visualize_stress(mof_cut, rows=f_mof, cols=t_mof+15, smooth=False, xlim=(16.2, 16.5), ylim=(-700,700), save_fig=True)

Applied stress energy cut:  1000
		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [62]:
tmp_t_prime, tmp_xi_prime = invert_wigner_function_re(S_mat=mof_cut, frequency_array=f_mof, time_array=t_mof, debug_plot=False)

In [20]:
# save the results:
np.savetxt("../../data/data_txt/waveform_from_inverted_wigner.txt", tmp_xi_prime)
np.savetxt("../../data/data_txt/times_from_inverted_wigner.txt", tmp_t_prime)

In [17]:
plt.plot(tmp_t_prime, tmp_xi_prime)

In [76]:
plt.plot(tmp_t_prime+16, tmp_xi_prime/np.max(tmp_xi_prime), label=r"$S^{-1}$ of smoothed Wigner")
plt.plot(reconstructed_times+16, reconstructed_nrt/max(reconstructed_nrt), label=r" $S^{-1}$ of num. relativity template")
usual_plot(yl="Amplitude (normalized)", xlim=(16.21, 16.52), save_fig=True)

In [542]:
pa_tmp_k, pa_tmp_ps = power_analyze_re(tmp_t_prime, tmp_xi_prime*np.max(reconstructed_nrt)/np.max(tmp_xi_prime))

In [546]:
def smooth_loglog_freq(x, y, win=101):
    import numpy as np
    from scipy.signal import savgol_filter

    mask = (x != 0) & (y > 0)
    y_out = y.copy()
    y_out[mask] = 10**savgol_filter(np.log10(y[mask]), win, 1)
    return y_out

pa_tmp_ps_smoothed = smooth_loglog_freq(np.array(pa_tmp_k), np.array(pa_tmp_ps), win=10)
plt.plot(k_nrt, ps_nrt)
plt.plot(pa_tmp_k, pa_tmp_ps_smoothed)
# plt.plot(pa_tmp_k, pa_tmp_ps)
plt.loglog()
plt.show()